###### December 2019 - Roger Melko

Diagonalizing the Hamiltonian matrix for the transverse field Ising model to find the energy eigenvalues and eigenkets.   Calculate the groundstate magnetization.

We will use the same Hamiltonian convention as the QMC program:
$$
H = -J\sum_{\langle i j \rangle} \sigma^z_i \sigma^z_j - B \sum_i \sigma^x_i - h \sum_i \sigma^z_i
$$
where ${\bf \sigma}_i$ are Pauli operators.  In this convention, the 1+1 CFT is at $B/J = 1$ and $h=0$.

In [2]:
using LinearAlgebra

In [14]:
N = 6
Dim = 2^N

J = 1. #exchange interaction
B = 1. #transverse field
h = 1. #longitudinal field

Hamiltonian = zeros(Dim,Dim)   #This is your 2D Hamiltonian matrix

Spin2 = 0 #give it a scope outside of the loop

for Ket = 0:Dim-1  #Loop over Hilbert Space
    Diagonal = 0.
    for SpinIndex = 0:N-2  #Loop over spin index (base zero, stop one spin before the end of the chain)
        Spin1 = 2*((Ket>>SpinIndex)&1) - 1
        NextIndex = SpinIndex + 1
        Spin2 = 2*((Ket>>NextIndex)&1) - 1
        Diagonal = Diagonal - J*Spin1*Spin2 - h*Spin1 #spins are +1 and -1
    end
    Diagonal = Diagonal - h*Spin2 #this is the spin at the end of the chain
    
    Hamiltonian[Ket+1,Ket+1] = Diagonal
    
    for SpinIndex = 0:N-1
        bit = 2^SpinIndex   #The "label" of the bit to be flipped
        Bra = Ket ⊻ bit    #Binary XOR flips the bit
        Hamiltonian[Bra+1,Ket+1] = -B
    end
end
Hamiltonian = Hermitian(Hamiltonian);

In the Julia LinearAlgebra package, the eigen function finds eigenvalues and eigenvectors.  They are ordered; i.e. the groundstate energy corresponds to index 1

In [10]:
Diag = eigen(Hamiltonian);

In [11]:
GroundState = Diag.vectors[:, 1];  #this gives the groundstate eigenvector
Diag.values[1] / N

-2.0254700931441834

In [12]:
##### Calculate the groundstate magnetization <m^2> in the Z direction
magnetization = zeros(Dim)
abs_mag = zeros(Dim)
mag_squared = zeros(Dim)

SumSz = dropdims(sum(@. (2 * (((0:Dim-1) >> (0:N-1)') & 1) - 1); dims=2); dims=2)
AbsSumSz = abs.(SumSz)
SumSzSq = abs2.(SumSz)

magnetization = SumSz' * abs2.(Diag.vectors)
abs_mag = AbsSumSz' * abs2.(Diag.vectors)
mag_squared = SumSzSq' * abs2.(Diag.vectors)

(magnetization[1] / N), (abs_mag[1] / N), (mag_squared[1] / (N*N))

(0.9218169780744616, 0.9219161014766092, 0.8764061766053675)

In [13]:
beta_vals = [20,15,10,5,3,2,1,0.8,0.5,0.2]
ED =zeros(Float64,length(beta_vals))
idx = 1
for β in beta_vals
    weights = exp.(-β * Diag.values)
    Z = sum(weights)
    E = dot(Diag.values, weights) / (N*Z)
    C = (β^2 * ((dot(Diag.values .^2, weights) / Z) - (N*E)^2))
    # magnetization of thermal state
    M = dot(weights, magnetization) / (N*Z)
    M_abs = dot(weights, abs_mag) / (N*Z)
    M2 = dot(weights, mag_squared) / (N*N*Z)
    println(β," ",E," ",C," ",M," ",M_abs," ",M2)
    #ED[idx,1] = β
    ED[idx,1] = M2
    idx += 1
end

20.0 -2.0254700931441834 0.0 0.9218169780744615 0.9219161014766091 0.8764061766053675
15.0 -2.0254700931441834 0.0 0.9218169780744616 0.9219161014766092 0.8764061766053675
10.0 -2.0254700931441834 0.0 0.9218169780744617 0.9219161014766092 0.8764061766053673
5.0 -2.0254700918398214 8.142706064973027e-7 0.9218169773870775 0.9219161007970696 0.8764061756382858
3.0 -2.025464556556553 0.001258818861117561 0.92181406555675 0.9219132244461387 0.876402090530229
2.0 -2.0250758468165397 0.041166134265040455 0.9216112382527073 0.921713255386069 0.8761194715564993
1.0 -1.9878286983622329 1.0997361215924286 0.9027884075697914 0.9034685997171946 0.8509112010838368
0.8 -1.9242978816996115 1.9447361213311891 0.8711339087916541 0.8736502359135505 0.8107309732794019
0.5 -1.5806158835777881 3.107977350815542 0.7016998444467872 0.7267649299575736 0.6228443257167531
0.2 -0.6555678577972203 0.8632841666647663 0.26504896391313437 0.42826645843110894 0.28014624204400584
